In [1]:
import sys
!{sys.executable} -m pip install geopandas shapely pyproj fiona

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np

In [3]:
df = pd.read_csv("/green-projects/project-urban_colocation_intelligence/workspace/share/data/locomizer_filtered_output/canarywharf_r10_2025-03_04.csv")

df.head()

,id,lat,lon,ts,date,gridcell,type
0,3fe1f077e29799ec1b28a80d42304b8f,51.507560,-0.024580,1743382001,2025-03-31,8a194ad266d7fff,stop
1,5549c0383dd1440e3288f4b1c18f750d,51.507626,-0.025254,1743365709,2025-03-30,8a194ad266d7fff,pedestrian
2,cbd87fa1a40281775f4655f875103f54,51.507140,-0.024240,1743082018,2025-03-27,8a194ad266d7fff,pedestrian
3,c1a1163c2b9defb34b767b4e440916a8,51.507236,-0.025280,1742873344,2025-03-25,8a194ad266d7fff,stop
4,a18d4344aada48028aa2a4330c1ae0a8,51.507580,-0.024490,1743019306,2025-03-26,8a194ad266d7fff,pedestrian


In [4]:
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326"
)

gdf = gdf.to_crs(epsg=27700)

gdf.head()

,id,lat,lon,ts,date,gridcell,type,geometry
0,3fe1f077e29799ec1b28a80d42304b8f,51.507560,-0.024580,1743382001,2025-03-31,8a194ad266d7fff,stop,POINT (537191.194 180586.173)
1,5549c0383dd1440e3288f4b1c18f750d,51.507626,-0.025254,1743365709,2025-03-30,8a194ad266d7fff,pedestrian,POINT (537144.224 180592.25)
2,cbd87fa1a40281775f4655f875103f54,51.507140,-0.024240,1743082018,2025-03-27,8a194ad266d7fff,pedestrian,POINT (537216.048 180540.105)
3,c1a1163c2b9defb34b767b4e440916a8,51.507236,-0.025280,1742873344,2025-03-25,8a194ad266d7fff,stop,POINT (537143.59 180548.833)
4,a18d4344aada48028aa2a4330c1ae0a8,51.507580,-0.024490,1743019306,2025-03-26,8a194ad266d7fff,pedestrian,POINT (537197.379 180588.566)


In [5]:
gdf["datetime"] = pd.to_datetime(gdf["ts"], unit="s")

In [6]:
gdf["hour"] = gdf["datetime"].dt.hour
gdf["weekday"] = gdf["datetime"].dt.weekday

In [7]:
gdf[["datetime", "hour", "weekday"]].head(10)

,datetime,hour,weekday
0,2025-03-31 00:46:41,0,0
1,2025-03-30 20:15:09,20,6
2,2025-03-27 13:26:58,13,3
3,2025-03-25 03:29:04,3,1
4,2025-03-26 20:01:46,20,2
5,2025-03-30 11:26:06,11,6
6,2025-03-29 05:56:01,5,5
7,2025-03-29 08:40:26,8,5
8,2025-03-27 05:29:51,5,3
9,2025-03-26 05:19:28,5,2


In [8]:
work_gdf = gdf[
    (gdf["weekday"] < 5) &   # Mon–Fri
    (gdf["hour"] >= 8) &
    (gdf["hour"] <= 18)
].copy()

print("Filtered points:", len(work_gdf))

Filtered points: 1980693


In [13]:
work_gdf["hour_bin"] = work_gdf["datetime"].dt.floor("h") 

work_gdf = (
    work_gdf
    .sort_values("datetime")
    .drop_duplicates(subset=["id", "hour_bin"])  
)

print("After dedup:", len(work_gdf))
print("Unique users:", work_gdf["id"].nunique())

After dedup: 877957
Unique users: 261553


**worker-poi match**

In [14]:
work_gdf["date"] = work_gdf["datetime"].dt.date

user_days = (
    work_gdf.groupby("id")["date"]
    .nunique()
    .reset_index()
)

user_days.columns = ["id", "active_days"]

In [15]:
likely_workers = user_days[user_days["active_days"] >= 3]

In [16]:
worker_ids = set(likely_workers["id"])

worker_gdf = work_gdf[work_gdf["id"].isin(worker_ids)].copy()

print("Likely workers:", len(worker_ids))
print("Worker points:", len(worker_gdf))

Likely workers: 37232
Worker points: 457793


In [17]:
worker_gdf.crs

<Projected CRS: EPSG:27700>
Name: OSGB36 / British National Grid
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: United Kingdom (UK) - offshore to boundary of UKCS within 49°45'N to 61°N and 9°W to 2°E; onshore Great Britain (England, Wales and Scotland). Isle of Man onshore.
- bounds: (-9.01, 49.75, 2.01, 61.01)
Coordinate Operation:
- name: British National Grid
- method: Transverse Mercator
Datum: Ordnance Survey of Great Britain 1936
- Ellipsoid: Airy 1830
- Prime Meridian: Greenwich

In [18]:
poi_gdf = gpd.read_file("/green-projects/project-urban_colocation_intelligence/workspace/share/data/poi_within_canary_wharf.geojson")

poi_gdf = poi_gdf.to_crs(epsg=27700)

# Convert polygons → points
poi_gdf["geometry"] = poi_gdf.geometry.centroid

In [20]:
# Remove leftover join columns if they exist
for col in ["index_right", "index_left"]:
    if col in worker_gdf.columns:
        worker_gdf = worker_gdf.drop(columns=[col])
    if col in poi_gdf.columns:
        poi_gdf = poi_gdf.drop(columns=[col])

In [21]:
worker_gdf = worker_gdf.reset_index(drop=True)
poi_gdf = poi_gdf.reset_index(drop=True)

In [22]:
matched = gpd.sjoin_nearest(
    worker_gdf,
    poi_gdf,
    how="left",
    distance_col="dist"
)

matched = matched[matched["dist"] <= 80]

print("Matched records:", len(matched))

Matched records: 442022


In [23]:
matched["CATEGORY_TAGS"].value_counts().head(20)

CATEGORY_TAGS
[Street Food]                                                                                         163507
[]                                                                                                    113480
[Metro Station, Train Station]                                                                          8048
[Corporate Offices]                                                                                     6461
[Property Management]                                                                                   4903
[Cafe, Healthy Food, Smoothie & Juice Bar]                                                              3824
[Shopping Mall]                                                                                         3777
[Hotels, Motels]                                                                                        3678
[Bakery, Snacks, Tea House]                                                                             3270
[Inve

In [27]:
def get_primary_category(x):
    if isinstance(x, list) and len(x) > 0:
        return x[0]
    return "Unknown"

matched["category_clean"] = matched["CATEGORY_TAGS"].apply(get_primary_category)

In [28]:
category_workers = (
    matched.groupby("category_clean")["id"]
    .nunique()
    .sort_values(ascending=False)
)

In [29]:
category_workers = category_workers.to_frame(name="worker_count")

category_workers["share"] = (
    category_workers["worker_count"] / worker_gdf["id"].nunique()
)

In [30]:
category_workers.head(10)

,worker_count,share
category_clean,,
Street Food,16932,0.454770
Unknown,16072,0.431672
Bakery,4313,0.115841
Bar or Pub,4212,0.113128
Metro Station,3485,0.093602
American Food,3261,0.087586
Asian Food,2690,0.072250
Accessories,2436,0.065428
Cafe,2424,0.065105
